In [1]:
# =========================
# INSTALL
# =========================
!pip install Sastrawi scikit-learn

# =========================
# IMPORT
# =========================
import pandas as pd
import re
import pickle
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# =========================
# LOAD DATA
# =========================
path = "/kaggle/input/notebooks/mengkoding47/crawling-ferischa/detik_random_1260.csv"
df = pd.read_csv(path)

print("Data awal:", df.shape)

# =========================
# INIT NLP
# =========================
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

# =========================
# PREPROCESS
# =========================
def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    tokens = [stemmer.stem(t) for t in tokens]

    return " ".join(tokens)

print("Processing...")

df["isi_clean"] = df["isi"].apply(preprocess_text)
df["judul_clean"] = df["judul"].apply(preprocess_text)
df["text_clean"] = df["judul_clean"] + " " + df["isi_clean"]

# =========================
# TF-IDF (SPARSE MATRIX)
# =========================
vectorizer = TfidfVectorizer(max_features=5000)

tfidf_matrix = vectorizer.fit_transform(df["text_clean"])

print("TF-IDF shape:", tfidf_matrix.shape)

# =========================
# SIMPAN UNTUK SEARCH ENGINE
# =========================

# 1. simpan metadata saja (tanpa vector)
df[["judul", "link", "kategori"]].to_csv(
    "/kaggle/working/articles_metadata.csv",
    index=False
)

# 2. simpan TF-IDF matrix (SPARSE)
with open("/kaggle/working/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

# 3. simpan vectorizer
with open("/kaggle/working/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Selesai simpan semua file!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.5 MB/s eta 0:00:0000:01
Data awal: (1260, 4)
Processing...
TF-IDF shape: (1260, 5000)
Selesai simpan semua file!
